# Baseline TF-IDF + Naive Bayes

Este es el único experimento de modelo que se conserva en el repositorio. Trabaja a nivel de noticia y usa únicamente `title + body`. Claims, evidencia, fuente, país y similitudes quedan fuera de las features para evitar fuga de información.

El notebook no asigna etiquetas ni convierte un export de Label Studio directamente en verdad. Solo consume el CSV binario Gold después de la revisión humana.

## Dónde va el Gold CSV

Descarga desde la consola de OCI el objeto que publica la Function después de la revisión humana:

```text
gold/training/training_<YYYYMMDD>.csv
```

Copia ese único archivo localmente en `gold/training/` del repositorio. Este notebook no convierte exports de Label Studio, no une Body con Evidence y no asigna etiquetas: la Function de OCI ya construye y valida el Gold. Si cambias la ruta o el nombre, ajusta `GOLD_CSV` en la siguiente celda.

In [ ]:
from collections import Counter
import csv
import json
from pathlib import Path

import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Ejecuta el notebook desde la raíz del repositorio o cambia esta ruta.
GOLD_CSV = Path('gold/training/training_20261019.csv')
RUN_DIR = Path('artifacts/training_runs/notebook_tfidf_naive_bayes')
SEED = 42
MIN_PER_CLASS = 10
LABELS = ['REFUTADA', 'RESPALDADA']

print(f'Entrada: {GOLD_CSV.resolve()}')
print(f'Salida: {RUN_DIR.resolve()}')

In [ ]:
def _clean(value):
    return str(value or '').strip()


def load_binary_rows(path):
    if not path.is_file():
        raise FileNotFoundError(
            f'No existe {path}. Primero genera Gold con Body + Evidence + Label Studio.'
        )
    rows = []
    seen = set()
    with path.open('r', encoding='utf-8-sig', newline='') as handle:
        reader = csv.DictReader(handle)
        required = {'record_id', 'title', 'body', 'label'}
        missing = required - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f'Faltan columnas: {sorted(missing)}')
        for line, source in enumerate(reader, start=2):
            row = {key: _clean(value) for key, value in source.items() if key}
            record_id = row.get('record_id', '')
            if not record_id or record_id in seen:
                raise ValueError(f'record_id vacío o repetido en línea {line}: {record_id!r}')
            seen.add(record_id)
            if row.get('label') not in LABELS:
                continue  # NO_DETERMINABLE y EXCLUIDA quedan fuera del entrenamiento binario
            if not row.get('title') or not row.get('body'):
                raise ValueError(f'{record_id} no tiene title o body completo')
            rows.append(row)
    return rows


rows = load_binary_rows(GOLD_CSV)
class_counts = Counter(row['label'] for row in rows)
print(f'Noticias binarias: {len(rows)}')
print(f'Distribución: {dict(class_counts)}')
if set(class_counts) != set(LABELS):
    raise ValueError('Se necesitan las dos clases: REFUTADA y RESPALDADA.')
if min(class_counts.values()) < MIN_PER_CLASS:
    raise ValueError(
        f'Hay muy pocas noticias de una clase ({dict(class_counts)}). '
        f'Completa al menos {MIN_PER_CLASS} por clase antes de entrenar.'
    )

In [ ]:
def article_text(row):
    return f"{row['title']}\n\n{row['body']}"

train_rows, remainder_rows = train_test_split(
    rows, test_size=0.30, random_state=SEED,
    stratify=[row['label'] for row in rows],
)
validation_rows, test_rows = train_test_split(
    remainder_rows, test_size=0.50, random_state=SEED,
    stratify=[row['label'] for row in remainder_rows],
)

model = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True, strip_accents='unicode', ngram_range=(1, 2),
        sublinear_tf=True, min_df=1,
    )),
    ('classifier', MultinomialNB()),
])
model.fit([article_text(row) for row in train_rows], [row['label'] for row in train_rows])

print({
    'train': len(train_rows),
    'validation': len(validation_rows),
    'test': len(test_rows),
})

In [ ]:
def evaluate(split_name, split_rows):
    texts = [article_text(row) for row in split_rows]
    y_true = [row['label'] for row in split_rows]
    y_pred = model.predict(texts)
    report = classification_report(
        y_true, y_pred, labels=LABELS, target_names=LABELS,
        output_dict=True, zero_division=0,
    )
    print(f'\n{split_name}')
    print(classification_report(y_true, y_pred, labels=LABELS, target_names=LABELS, zero_division=0))
    print('Matriz de confusión [REFUTADA, RESPALDADA]:')
    print(confusion_matrix(y_true, y_pred, labels=LABELS))
    return report, y_pred

metrics = {}
predictions = []
for split_name, split_rows in [('train', train_rows), ('validation', validation_rows), ('test', test_rows)]:
    report, y_pred = evaluate(split_name, split_rows)
    metrics[split_name] = report
    for row, predicted in zip(split_rows, y_pred):
        predictions.append({
            'record_id': row['record_id'],
            'dataset_split': split_name,
            'true_label': row['label'],
            'predicted_label': predicted,
            'correct': str(int(row['label'] == predicted)),
        })

In [ ]:
RUN_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(model, RUN_DIR / 'model.joblib')

split_rows = []
for split_name, split in [('train', train_rows), ('validation', validation_rows), ('test', test_rows)]:
    split_rows.extend({
        'record_id': row['record_id'],
        'dataset_split': split_name,
        'label': row['label'],
    } for row in split)

with (RUN_DIR / 'split_manifest.csv').open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['record_id', 'dataset_split', 'label'])
    writer.writeheader()
    writer.writerows(split_rows)

with (RUN_DIR / 'predictions.csv').open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=predictions[0].keys())
    writer.writeheader()
    writer.writerows(predictions)

summary = {
    'model': 'tfidf_multinomial_nb',
    'input_csv': str(GOLD_CSV),
    'rows_total': len(rows),
    'class_counts': dict(class_counts),
    'split_counts': {'train': len(train_rows), 'validation': len(validation_rows), 'test': len(test_rows)},
    'features': ['title', 'body'],
    'seed': SEED,
    'metrics': metrics,
}
(RUN_DIR / 'metrics.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(f'Artefactos guardados en {RUN_DIR.resolve()}')